In [8]:
"""
==============================================================
DATA PREPARATION - Deteksi Stunting Posyandu
==============================================================
Target   : Status Stunting + Status Gizi
Model    : Random Forest Classifier
==============================================================
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ──────────────────────────────────────────────
# FASE 1: LOAD DATA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA")
print("=" * 55)

df = pd.read_csv('../data/raw/dataset_gabungan2.csv')
print(f"✅ Data berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
print(f"   Kolom: {df.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 2: DROP KOLOM TIDAK DIPERLUKAN
# ──────────────────────────────────────────────
# print("=" * 55)
# print("FASE 2: DROP KOLOM TIDAK DIPERLUKAN")
# print("=" * 55)

# drop_cols = ['Nama', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'LiLA', 'Status Gizi']
# df.drop(columns=drop_cols, inplace=True)
# print(f"✅ Kolom di-drop: {drop_cols}")
# print(f"   Sisa kolom: {df.columns.tolist()}\n")


# FASE 3: CEK KOLOM USIA (sudah dalam bulan)
print("=" * 55)
print("FASE 3: CEK KOLOM USIA")
print("=" * 55)

# Usia sudah dalam format bulan, tidak perlu konversi
print("Distribusi usia (dalam bulan) sebelum filter:")
print(df['Usia_Bulan'].value_counts().sort_index().to_string())
print()


# ──────────────────────────────────────────────
# FASE 4: FILTER HANYA BALITA (0-60 BULAN)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: FILTER BALITA (USIA ≤ 60 BULAN)")
print("=" * 55)

sebelum = len(df)
df = df[df['Usia_Bulan'] <= 60].copy()
sesudah = len(df)
print(f"✅ Data sebelum filter : {sebelum} baris")
print(f"✅ Data setelah filter : {sesudah} baris")
print(f"   Dibuang            : {sebelum - sesudah} baris (usia > 60 bulan)\n")


# ──────────────────────────────────────────────
# FASE 5: BERSIHKAN KOLOM JK
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: BERSIHKAN KOLOM JENIS KELAMIN")
print("=" * 55)

print(f"   Nilai unik sebelum : {df['JK'].unique()}")
df['JK'] = df['JK'].str.strip()
print(f"   Nilai unik sesudah : {df['JK'].unique()}\n")


# ──────────────────────────────────────────────
# FASE 6: BUAT LABEL STATUS STUNTING (dari ZS TB/U)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: BUAT LABEL STATUS STUNTING")
print("=" * 55)

def buat_status_stunting(zs_tbu):
    """
    Klasifikasi stunting berdasarkan standar WHO:
    ZS TB/U < -3         : Sangat Pendek
    -3 <= ZS TB/U < -2   : Pendek
    ZS TB/U >= -2        : Normal
    """
    if pd.isna(zs_tbu):
        return np.nan
    elif zs_tbu < -3:
        return 'Sangat Pendek'
    elif zs_tbu < -2:
        return 'Pendek'
    else:
        return 'Normal'

df['Status_Stunting'] = df['ZS TB/U'].apply(buat_status_stunting)
print("✅ Distribusi Status Stunting:")
print(df['Status_Stunting'].value_counts().to_string())
print()


# ──────────────────────────────────────────────
# FASE 7: TANGANI MISSING VALUES
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: TANGANI MISSING VALUES")
print("=" * 55)

print("Missing values sebelum ditangani:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())
print()

# Drop hanya baris yang Status_Stunting kosong (ZS TB/U kosong)
sebelum = len(df)
df.dropna(subset=['Status_Stunting'], inplace=True)
sesudah = len(df)
print(f"✅ Baris dibuang karena ZS TB/U kosong: {sebelum - sesudah}")

# Drop kolom ZS setelah selesai dipakai
df.drop(columns=['ZS BB/U', 'ZS TB/U', 'ZS BB/TB'], inplace=True)

print(f"\nMissing values setelah ditangani:")
print(df.isnull().sum().to_string())
print()

# Drop kolom ZS setelah selesai dipakai
# df.drop(columns=['ZS BB/U', 'ZS TB/U', 'ZS BB/TB'], inplace=True)

# print(f"✅ Baris dibuang karena ZS TB/U kosong: {sebelum - sesudah}")
# print(f"\nMissing values setelah ditangani:")
# print(df.isnull().sum().to_string())

# # ──────────────────────────────────────────────
# # FASE 7.2: UNDERSAMPLE KELAS NORMAL
# # ──────────────────────────────────────────────
# print("=" * 55)
# print("FASE 7.2: UNDERSAMPLE KELAS NORMAL")
# print("=" * 55)

# df_normal        = df[df['Status_Stunting'] == 'Normal'].sample(n=1000, random_state=42)
# df_pendek        = df[df['Status_Stunting'] == 'Pendek']
# df_sangat_pendek = df[df['Status_Stunting'] == 'Sangat Pendek']

# df = pd.concat([df_normal, df_pendek, df_sangat_pendek]).sample(frac=1, random_state=42).reset_index(drop=True)

# print(f"✅ Distribusi setelah undersample:")
# for label, count in df['Status_Stunting'].value_counts().items():
#     pct = count / len(df) * 100
#     bar = 'X' * int(pct / 2)
#     print(f"   {label:<15}: {count:>4} ({pct:.1f}%) {bar}")
# print(f"\n   Total data: {len(df)} baris\n")

# ──────────────────────────────────────────────
# FASE 7.5: TAMBAH FITUR TURUNAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7.5: TAMBAH FITUR TURUNAN")
print("=" * 55)

# BMI = Berat (kg) / Tinggi (m)²
df['BMI'] = df['Berat'] / ((df['Tinggi'] / 100) ** 2)

# Rasio BB/TB
df['Rasio_BB_TB'] = df['Berat'] / df['Tinggi']

# Cek outlier sebelum filter
print(f"Sebelum filter outlier: {len(df)} baris")
print(f"BMI    : min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")
print(f"Tinggi : min={df['Tinggi'].min():.2f}, max={df['Tinggi'].max():.2f}")
print(f"Berat  : min={df['Berat'].min():.2f}, max={df['Berat'].max():.2f}")

# Filter nilai tidak wajar
df = df[df['Tinggi'] >= 40].copy()   # Tinggi minimal 40cm
df = df[df['Tinggi'] <= 120].copy()  # Tinggi maksimal 120cm
df = df[df['Berat'] >= 1].copy()     # Berat minimal 1kg
df = df[df['Berat'] <= 30].copy()    # Berat maksimal 30kg
df = df[df['BMI'] <= 40].copy()      # BMI maksimal wajar

print(f"Setelah filter outlier : {len(df)} baris")
print(f"BMI    : min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")

print("✅ Fitur turunan ditambahkan:")
print(f"   BMI         : mean={df['BMI'].mean():.2f}, min={df['BMI'].min():.2f}, max={df['BMI'].max():.2f}")
print(f"   Rasio_BB_TB : mean={df['Rasio_BB_TB'].mean():.2f}, min={df['Rasio_BB_TB'].min():.2f}, max={df['Rasio_BB_TB'].max():.2f}")
print()

# ──────────────────────────────────────────────
# FASE 8: ENCODING VARIABEL KATEGORIKAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 8: ENCODING VARIABEL KATEGORIKAL")
print("=" * 55)

# Encode JK: L=1, P=0
df['JK_encoded'] = df['JK'].map({'L': 1, 'P': 0})
print("✅ JK encoded: L=1, P=0")

# Urutan alfabetis LabelEncoder: Normal=0, Pendek=1, Sangat Pendek=2
le_stunting = LabelEncoder()
df['Status_Stunting_encoded'] = le_stunting.fit_transform(df['Status_Stunting'])
print(f"✅ Status Stunting encoded:")
for label, kode in zip(le_stunting.classes_, le_stunting.transform(le_stunting.classes_)):
    print(f"   {label:<15} = {kode}")
print()


# ──────────────────────────────────────────────
# FASE 9: SUSUN DATASET FINAL
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 9: SUSUN DATASET FINAL")
print("=" * 55)

# Fitur (X) sebagai pembantu
feature_cols = ['JK_encoded', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Rasio_BB_TB']

# Target stunting
df_clean = df[feature_cols + ['Status_Stunting']].copy()
df_clean.columns = ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'BMI', 'Rasio_BB_TB', 'Status_Stunting']
 
print(f"✅ Fitur  : {['JK', 'Usia_Bulan', 'Berat', 'Tinggi']}")
print(f"✅ Target : Status_Stunting")
print(f"   Kelas  : {le_stunting.classes_.tolist()}")
print(f"\nShape dataset final: {df_clean.shape}")
print()


# ──────────────────────────────────────────────
# FASE 10: CEK CLASS IMBALANCE
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 10: CEK CLASS IMBALANCE")
print("=" * 55)
 
print("Distribusi Status_Stunting:")
for label, count in df_clean['Status_Stunting'].value_counts().items():
    pct = count / len(df_clean) * 100
    bar = 'X' * int(pct / 2)
    print(f"   {label:<15}: {count:>4} ({pct:.1f}%) {bar}")
print()
 
min_pct = df_clean['Status_Stunting'].value_counts(normalize=True).min() * 100
if min_pct < 15:
    print("PERINGATAN: Ada class imbalance! Gunakan class_weight='balanced' saat training RF.")
else:
    print("✅ Distribusi kelas cukup seimbang.")
print()

# ──────────────────────────────────────────────
# FASE 11: SIMPAN DATASET BERSIH
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 11: SIMPAN DATASET")
print("=" * 55)

# Dataset lengkap bersih (dengan label asli untuk referensi)
df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print("✅ dataset_clean.csv tersimpan!")
print(f"   Berisi {len(df_clean)} baris dan {len(df_clean.columns)} kolom")
print(f"   Kolom: {df_clean.columns.tolist()}")
print()

# ──────────────────────────────────────────────
# RINGKASAN AKHIR
# ──────────────────────────────────────────────
print("=" * 55)
print("✅ DATA PREPARATION SELESAI")
print("=" * 55)
print(f"   Data awal           : 1351 baris")
print(f"   Data setelah proses : {len(df_clean)} baris")
print(f"   Jumlah fitur        : 4")
print(f"   Target              : Status Stunting (3 kelas)")
print(f"   Mapping label       : Normal=0 | Pendek=1 | Sangat Pendek=2")
print(f"   Siap untuk training Random Forest!")
print("=" * 55)

FASE 1: LOAD DATA
✅ Data berhasil dimuat: 2725 baris, 7 kolom
   Kolom: ['JK', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB', 'Usia_Bulan']

FASE 3: CEK KOLOM USIA
Distribusi usia (dalam bulan) sebelum filter:
Usia_Bulan
0       4
1      33
2      17
3      16
4      18
5      21
6      30
7      24
8      23
9      29
10     29
11     16
12     22
13     25
14     21
15     14
16     13
17     22
18     23
19     26
20     20
21     19
22     18
23     28
24     83
25     24
26     16
27     21
28     31
29     37
30     24
31     26
32     17
33     22
34     28
35     40
36    231
37     33
38     34
39     16
40     25
41     13
42     37
43     36
44     27
45     27
46     30
47     32
48    342
49     29
50     28
51     25
52     27
53     32
54     20
55     23
56     22
57     21
58     16
59     14
60    328
72    232
84    133
96     12

FASE 4: FILTER BALITA (USIA ≤ 60 BULAN)
✅ Data sebelum filter : 2725 baris
✅ Data setelah filter : 2348 baris
   Dibuang            